# Transformer Forecaster — CPU Edition

Predice **1 semana** (`144 × 7 = 1.008` pasos de 10 min) de consumo eléctrico.
Diseñado para entrenar en **CPU** en tiempo razonable (~5-15 min total).

### Por qué las versiones anteriores eran lentas
| Problema | Solución aplicada aquí |
|---|---|
| Atención O(n²) con n=1.008 | SEQ_LEN = **144 pasos** (1 día exacto, 10 min/muestra) |
| Decoder autoregresivo: 1.008 forward passes por muestra | **Encoder-only**: 1 solo forward pass predice toda la semana |
| Secuencias largas en batches grandes | BATCH=32, secuencias cortas → muy rápido |

### ¿Es válido usar solo encoder?
Sí. El decoder autoregresivo es necesario cuando generas texto token a token.
Para forecasting de longitud fija, el encoder lee la historia y una cabeza
lineal proyecta directamente a los `pred_len` pasos futuros. Es el enfoque
de modelos como **PatchTST** o **Autoformer**.


## 1. Imports

In [2]:
import numpy as np
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import os
import pandas as pd

## 2. Configuración

In [ ]:
FREQ     = 144           # Muestras por día (cada 10 min → 6×24)
PRED_LEN = FREQ * 7      # Predecir 1 semana = 1.008 pasos  ← OBLIGATORIO

# ── CLAVE: ventana de entrada muy corta ──────────────────────────────────────

SEQ_LEN  = FREQ * 1      # 1 día exacto de historia = 144 pasos (una muestra cada 10 min)

EPOCHS     = 20
BATCH      = 32          # Batches más grandes → gradientes más estables, mismo tiempo
LR         = 1e-3
MODEL_PATH = "transformer_cpu.pth"
DEVICE     = "cpu"       # Forzamos CPU explícitamente

# ── Arquitectura ─────────────────────────────────────────────────────────────
D_MODEL         = 32
NHEAD           = 4      # D_MODEL debe ser divisible por NHEAD
ENC_LAYERS      = 2
DIM_FEEDFORWARD = 64
DROPOUT         = 0.1

print(f"SEQ_LEN={SEQ_LEN} | PRED_LEN={PRED_LEN} | BATCH={BATCH} | DEVICE={DEVICE}")

SEQ_LEN=144 | PRED_LEN=1008 | BATCH=32 | DEVICE=cpu


## 3. Datos y normalización

In [4]:
data = pd.read_csv("../powerconsumption.csv")
data["Datetime"] = pd.to_datetime(data["Datetime"])
data = data.sort_values("Datetime").set_index("Datetime")

series = data["PowerConsumption_Zone1"].dropna().values.astype("float32")

# Normalización Min-Max — fit solo sobre train
split = int(len(series) * 0.8)
s_min = series[:split].min()
s_max = series[:split].max()
series_norm = (series - s_min) / (s_max - s_min)

train_data = series_norm[:split]
test_data  = series_norm[split:]

print(f"Train: {len(train_data):,} | Test: {len(test_data):,} muestras")

Train: 41,932 | Test: 10,484 muestras


## 4. Dataset

In [5]:
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_len, pred_len):
        self.data     = data
        self.seq_len  = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return (
            torch.tensor(x, dtype=torch.float32).unsqueeze(-1),  # (seq_len, 1)
            torch.tensor(y, dtype=torch.float32).unsqueeze(-1),  # (pred_len, 1)
        )

train_ds = TimeSeriesDataset(train_data, SEQ_LEN, PRED_LEN)
test_ds  = TimeSeriesDataset(test_data,  SEQ_LEN, PRED_LEN)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False)

print(f"Train batches: {len(train_dl)} | Test batches: {len(test_dl)}")

Train batches: 1275 | Test batches: 292


## 5. Modelo — Transformer Encoder-Only

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])


class TransformerEncoderForecaster(nn.Module):
    """
    Transformer ENCODER-ONLY para forecasting de longitud fija.

    Flujo:
      (B, seq_len, 1)
        → proyección lineal → (B, seq_len, d_model)
        → positional encoding
        → N capas de TransformerEncoderLayer  ← solo atención self (sin decoder)
        → flatten del token [CLS] / promedio
        → cabeza lineal → (B, pred_len)
    """

    def __init__(self, seq_len, pred_len, d_model=D_MODEL, nhead=NHEAD,
                 num_layers=ENC_LAYERS, dim_feedforward=DIM_FEEDFORWARD,
                 dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        self.pos_enc    = PositionalEncoding(d_model, max_len=seq_len, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Cabeza de predicción: toma el promedio de todos los tokens
        # y proyecta directamente a pred_len valores
        self.head = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Linear(dim_feedforward, pred_len),
        )

    def forward(self, x):           # x: (B, seq_len, 1)
        x = self.pos_enc(self.input_proj(x))   # (B, seq_len, d_model)
        x = self.encoder(x)                    # (B, seq_len, d_model)
        x = x.mean(dim=1)                      # (B, d_model)  — pooling temporal
        return self.head(x).unsqueeze(-1)       # (B, pred_len, 1)


model = TransformerEncoderForecaster(seq_len=SEQ_LEN, pred_len=PRED_LEN)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {total:,}")

Parámetros entrenables: 84,784


## 6. Entrenamiento

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for x, y in loader:
        optimizer.zero_grad()
        pred = model(x)                   # (B, pred_len, 1)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total = 0.0
    for x, y in loader:
        total += criterion(model(x), y).item()
    return total / len(loader)


if os.path.exists(MODEL_PATH):
    print(f"Cargando pesos desde '{MODEL_PATH}'...")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
else:
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    criterion = nn.MSELoss()

    best_val = float("inf")
    history  = {"train": [], "val": []}

    for epoch in range(1, EPOCHS + 1):
        tr  = train_epoch(model, train_dl, optimizer, criterion)
        val = eval_epoch(model, test_dl, criterion)
        scheduler.step()
        history["train"].append(tr)
        history["val"].append(val)

        if val < best_val:
            best_val = val
            torch.save(model.state_dict(), MODEL_PATH)

        if epoch % 2 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{EPOCHS} | Train: {tr:.6f} | Val: {val:.6f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    print(f"\nMejor Val MSE: {best_val:.6f} → guardado en '{MODEL_PATH}'")

    plt.figure(figsize=(8, 3))
    plt.plot(history["train"], label="Train")
    plt.plot(history["val"],   label="Val")
    plt.xlabel("Epoch"); plt.ylabel("MSE")
    plt.title("Curva de pérdida"); plt.legend()
    plt.tight_layout(); plt.savefig("loss_curve.png", dpi=150); plt.show()

Epoch   1/20 | Train: 0.015088 | Val: 0.006223 | LR: 9.94e-04
Epoch   2/20 | Train: 0.004797 | Val: 0.008382 | LR: 9.76e-04
Epoch   4/20 | Train: 0.003929 | Val: 0.006826 | LR: 9.05e-04


## 7. Evaluación visual y métricas

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

x_test, y_test = test_ds[0]
with torch.no_grad():
    pred = model(x_test.unsqueeze(0)).squeeze().numpy()  # (pred_len,)

denorm = lambda v: v * (s_max - s_min) + s_min

y_true_raw = denorm(y_test.squeeze().numpy())
y_pred_raw = denorm(pred)

mae  = np.mean(np.abs(y_true_raw - y_pred_raw))
rmse = np.sqrt(np.mean((y_true_raw - y_pred_raw) ** 2))
mape = np.mean(np.abs((y_true_raw - y_pred_raw) / (y_true_raw + 1e-8))) * 100
print(f"MAE={mae:.1f} W | RMSE={rmse:.1f} W | MAPE={mape:.2f}%")

t_hist = np.arange(SEQ_LEN)
t_pred = np.arange(SEQ_LEN, SEQ_LEN + PRED_LEN)

plt.figure(figsize=(14, 4))
plt.plot(t_hist, denorm(x_test.squeeze().numpy()), color="steelblue", label="Historia", lw=0.8)
plt.plot(t_pred, y_true_raw, color="green",  linestyle="--", label="Real",      lw=1.2)
plt.plot(t_pred, y_pred_raw, color="tomato", linestyle=":",  label="Predicción", lw=1.4)
plt.axvline(SEQ_LEN, color="gray", lw=0.8, alpha=0.6)
plt.xlabel("Paso (10 min)"); plt.ylabel("Consumo (W)")
plt.title(f"Transformer Encoder-Only | MAE={mae:.1f} W  RMSE={rmse:.1f} W")
plt.legend(); plt.tight_layout()
plt.savefig("forecast_cpu.png", dpi=150); plt.show()